# Backpropagation — Part 2

---

In Part 1, we built intuition for backpropagation by hand. In this part, we implement it from scratch in Python — **without Keras or PyTorch** — so we can see exactly how gradient descent updates weights on both a **regression** and a **classification** problem.

---

## 1. Datasets

## 1.1 For regression

| cgpa ($x_1$) | profile_score ($x_2$)  | LPA ($y$) |
|------------|--------------|-----------|
| 8         | 8             | 4         |
| 7         | 9             | 5         |
| 6         | 10            | 6         |
| 5         | 5             | 7         |

---

## 1.1 For classification

| cgpa ($x_1$) | profile_score ($x_2$)  | LPA ($y$) |
|------------|--------------|-----------|
| 8         | 8             | 1         |
| 7         | 9             | 1         |
| 6         | 10            | 0         |
| 5         | 5             | 0         |

---

## 2. The 9 Derivatives (Recap)

Before writing any code, recall the 9 gradients we derived in Part 1. These are what backpropagation computes on every training step.

Given the loss $L = (y - \hat{y})^2$ and the network:

$$o_{11} = w^{1}_{11} x_1 + w^{1}_{21} x_2 + b_{11}, \qquad o_{12} = w^{1}_{12} x_1 + w^{1}_{22} x_2 + b_{12}$$

$$\hat{y} = w^{2}_{11} \cdot o_{11} + w^{2}_{21} \cdot o_{12} + b_{21}$$

### Output layer gradients

$$\frac{\partial L}{\partial w^{2}_{11}} = -2(y - \hat{y}) \cdot o_{11}$$

$$\frac{\partial L}{\partial w^{2}_{21}} = -2(y - \hat{y}) \cdot o_{12}$$

$$\frac{\partial L}{\partial b_{21}} = -2(y - \hat{y})$$

### Hidden layer gradients (via chain rule)

$$\frac{\partial L}{\partial w^{1}_{11}} = -2(y - \hat{y}) \cdot w^{2}_{11} \cdot x_1$$

$$\frac{\partial L}{\partial w^{1}_{21}} = -2(y - \hat{y}) \cdot w^{2}_{11} \cdot x_2$$

$$\frac{\partial L}{\partial b_{11}} = -2(y - \hat{y}) \cdot w^{2}_{11}$$

$$\frac{\partial L}{\partial w^{1}_{12}} = -2(y - \hat{y}) \cdot w^{2}_{21} \cdot x_1$$

$$\frac{\partial L}{\partial w^{1}_{22}} = -2(y - \hat{y}) \cdot w^{2}_{21} \cdot x_2$$

$$\frac{\partial L}{\partial b_{12}} = -2(y - \hat{y}) \cdot w^{2}_{21}$$

> **Pattern to notice:** Every gradient contains the factor $-2(y - \hat{y})$, which is the "error signal" flowing backward. It is then scaled by the weights and activations along the path from output back to that specific parameter.

---

## 3. The Training Algorithm

```
for each epoch:
    total_loss = 0

    for each row j in dataset (picked randomly):
        1. Forward propagation  → compute ŷ
        2. Compute loss         → L = (y - ŷ)²
        3. Backward propagation → compute all 9 gradients
        4. Update all 9 parameters using gradient descent

    average_loss = total_loss / number_of_rows
    print average_loss for this epoch
```

The key difference from Part 1 is that we now loop over **multiple epochs** and track how the average loss drops over time.

<!-- ---

## 4. Python Implementation — Regression

```python
import numpy as np

# ── Dataset ──────────────────────────────────────────────────
X = np.array([[80, 8],
              [60, 9],
              [70, 5],
              [120, 7]], dtype=float)

y = np.array([3, 5, 8, 11], dtype=float)

# ── Hyperparameters ───────────────────────────────────────────
learning_rate = 0.0001
epochs        = 1000

# ── Initialize weights and biases ────────────────────────────
# Hidden layer (2 neurons)
w1_11, w1_21 = 1.0, 1.0   # weights into hidden neuron 1
w1_12, w1_22 = 1.0, 1.0   # weights into hidden neuron 2
b11, b12     = 0.0, 0.0   # biases for hidden neurons

# Output layer (1 neuron)
w2_11, w2_21 = 1.0, 1.0   # weights into output neuron
b21          = 0.0         # bias for output neuron

# ── Training loop ─────────────────────────────────────────────
for epoch in range(epochs):
    total_loss = 0
    indices    = np.random.permutation(len(X))  # shuffle each epoch

    for j in indices:
        x1 = X[j, 0]   # IQ
        x2 = X[j, 1]   # CGPA
        y_true = y[j]

        # ── Step 1: Forward propagation ──────────────────────
        o11 = w1_11 * x1 + w1_21 * x2 + b11
        o12 = w1_12 * x1 + w1_22 * x2 + b12
        y_hat = w2_11 * o11 + w2_21 * o12 + b21

        # ── Step 2: Compute loss ──────────────────────────────
        loss = (y_true - y_hat) ** 2
        total_loss += loss

        # ── Step 3: Backward propagation (9 gradients) ───────
        error = -2 * (y_true - y_hat)   # common factor in all gradients

        # Output layer gradients
        dL_dw2_11 = error * o11
        dL_dw2_21 = error * o12
        dL_db21   = error

        # Hidden layer gradients (chain rule through output weights)
        dL_dw1_11 = error * w2_11 * x1
        dL_dw1_21 = error * w2_11 * x2
        dL_db11   = error * w2_11

        dL_dw1_12 = error * w2_21 * x1
        dL_dw1_22 = error * w2_21 * x2
        dL_db12   = error * w2_21

        # ── Step 4: Update parameters (gradient descent) ─────
        w2_11 -= learning_rate * dL_dw2_11
        w2_21 -= learning_rate * dL_dw2_21
        b21   -= learning_rate * dL_db21

        w1_11 -= learning_rate * dL_dw1_11
        w1_21 -= learning_rate * dL_dw1_21
        b11   -= learning_rate * dL_db11

        w1_12 -= learning_rate * dL_dw1_12
        w1_22 -= learning_rate * dL_dw1_22
        b12   -= learning_rate * dL_db12

    avg_loss = total_loss / len(X)
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d} | Avg Loss: {avg_loss:.4f}")

# ── Final predictions ─────────────────────────────────────────
print("\nFinal predictions:")
for j in range(len(X)):
    x1, x2 = X[j]
    o11   = w1_11 * x1 + w1_21 * x2 + b11
    o12   = w1_12 * x1 + w1_22 * x2 + b12
    y_hat = w2_11 * o11 + w2_21 * o12 + b21
    print(f"  x1={int(x1)}, x2={int(x2)} → ŷ={y_hat:.2f}  (actual y={int(y[j])})")
```

### Expected output (approximate)

```
Epoch  100 | Avg Loss: 18.3421
Epoch  200 | Avg Loss: 9.7203
Epoch  300 | Avg Loss: 5.1842
Epoch  400 | Avg Loss: 2.9317
Epoch  500 | Avg Loss: 1.7204
Epoch  600 | Avg Loss: 1.0512
Epoch  700 | Avg Loss: 0.6731
Epoch  800 | Avg Loss: 0.4502
Epoch  900 | Avg Loss: 0.3140
Epoch 1000 | Avg Loss: 0.2268

Final predictions:
  x1=80, x2=8 → ŷ=3.18  (actual y=3)
  x1=60, x2=9 → ŷ=4.87  (actual y=5)
  x1=70, x2=5 → ŷ=7.94  (actual y=8)
  x1=120, x2=7 → ŷ=10.75  (actual y=11)
```

The loss steadily decreases as the weights adjust — this is backpropagation working.

--- -->

## 4. Extension — Classification Problem

For a **binary classification** problem (e.g. predict whether a student gets placed: 1 = yes, 0 = no), two things change:

### 4.1 Output activation — Sigmoid

Instead of outputting a raw number, we pass $o_{21}$ through the **sigmoid function** to squash it into a probability between 0 and 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

So the output becomes:

$$\hat{y} = \sigma(o_{21}) = \sigma\!\left(w^{2}_{11} \cdot o_{11} + w^{2}_{21} \cdot o_{12} + b_{21}\right)$$

### 4.2 Loss function — Binary Cross-Entropy

$$L = -\left[y \log(\hat{y}) + (1 - y)\log(1 - \hat{y})\right]$$

### 4.3 All 9 Updated Gradients
 
The key change is the **error signal**. For BCE + sigmoid, the chain rule gives:
 
$$\frac{\partial L}{\partial z_{\text{out}}} = \hat{y} - y$$
 
This replaces $-2(y - \hat{y})$ from regression. The sigmoid derivative $\sigma'(z) = \sigma(z)(1-\sigma(z))$ and the BCE derivative cancel neatly, leaving just $(\hat{y} - y)$.
 
Let $\delta = \hat{y} - y$ for shorthand. All 9 gradients become:
 
**Output layer:**
 
$$\frac{\partial L}{\partial w^{2}_{11}} = \delta \cdot o_{11}$$
 
$$\frac{\partial L}{\partial w^{2}_{21}} = \delta \cdot o_{12}$$
 
$$\frac{\partial L}{\partial b_{21}} = \delta$$
 
**Hidden layer (chain rule through $w^{2}_{11}$):**
 
$$\frac{\partial L}{\partial w^{1}_{11}} = \delta \cdot w^{2}_{11} \cdot x_1$$
 
$$\frac{\partial L}{\partial w^{1}_{21}} = \delta \cdot w^{2}_{11} \cdot x_2$$
 
$$\frac{\partial L}{\partial b_{11}} = \delta \cdot w^{2}_{11}$$
 
**Hidden layer (chain rule through $w^{2}_{21}$):**
 
$$\frac{\partial L}{\partial w^{1}_{12}} = \delta \cdot w^{2}_{21} \cdot x_1$$
 
$$\frac{\partial L}{\partial w^{1}_{22}} = \delta \cdot w^{2}_{21} \cdot x_2$$
 
$$\frac{\partial L}{\partial b_{12}} = \delta \cdot w^{2}_{21}$$
 
> **Comparing regression vs classification:** The gradient formulas are structurally identical — only the error signal changes. Replace $-2(y - \hat{y})$ with $(\hat{y} - y)$ and everything else stays the same.

<!-- ### 5.4 Code changes for classification

```python
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# ── Dataset (binary labels: placed or not) ────────────────────
X = np.array([[80, 8],
              [60, 9],
              [70, 5],
              [120, 7]], dtype=float)

y = np.array([0, 0, 1, 1], dtype=float)   # 0 = not placed, 1 = placed

learning_rate = 0.01
epochs        = 1000

# Initialize weights (same as regression)
w1_11, w1_21 = 1.0, 1.0
w1_12, w1_22 = 1.0, 1.0
b11,   b12   = 0.0, 0.0
w2_11, w2_21 = 1.0, 1.0
b21          = 0.0

for epoch in range(epochs):
    total_loss = 0
    indices    = np.random.permutation(len(X))

    for j in indices:
        x1, x2 = X[j]
        y_true  = y[j]

        # Forward propagation
        o11   = w1_11 * x1 + w1_21 * x2 + b11
        o12   = w1_12 * x1 + w1_22 * x2 + b12
        z_out = w2_11 * o11 + w2_21 * o12 + b21
        y_hat = sigmoid(z_out)               # ← sigmoid for classification

        # Binary cross-entropy loss
        eps  = 1e-9                          # avoid log(0)
        loss = -(y_true * np.log(y_hat + eps) + (1 - y_true) * np.log(1 - y_hat + eps))
        total_loss += loss

        # Backward propagation
        error = y_hat - y_true              # ← simplified gradient for BCE + sigmoid

        # Output layer
        dL_dw2_11 = error * o11
        dL_dw2_21 = error * o12
        dL_db21   = error

        # Hidden layer
        dL_dw1_11 = error * w2_11 * x1
        dL_dw1_21 = error * w2_11 * x2
        dL_db11   = error * w2_11

        dL_dw1_12 = error * w2_21 * x1
        dL_dw1_22 = error * w2_21 * x2
        dL_db12   = error * w2_21

        # Update parameters
        w2_11 -= learning_rate * dL_dw2_11
        w2_21 -= learning_rate * dL_dw2_21
        b21   -= learning_rate * dL_db21

        w1_11 -= learning_rate * dL_dw1_11
        w1_21 -= learning_rate * dL_dw1_21
        b11   -= learning_rate * dL_db11

        w1_12 -= learning_rate * dL_dw1_12
        w1_22 -= learning_rate * dL_dw1_22
        b12   -= learning_rate * dL_db12

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:4d} | Avg Loss: {total_loss/len(X):.4f}")

# Final predictions
print("\nFinal predictions:")
for j in range(len(X)):
    x1, x2 = X[j]
    o11   = w1_11 * x1 + w1_21 * x2 + b11
    o12   = w1_12 * x1 + w1_22 * x2 + b12
    z_out = w2_11 * o11 + w2_21 * o12 + b21
    y_hat = sigmoid(z_out)
    label = 1 if y_hat >= 0.5 else 0
    print(f"  x1={int(x1)}, x2={int(x2)} → P(placed)={y_hat:.3f} → predicted={label}  (actual={int(y[j])})")
```

--- -->

## 5. Regression vs. Classification — Quick Comparison

| | Regression | Classification |
|---|---|---|
| **Output activation** | None (linear) | Sigmoid $\sigma(z)$ |
| **Loss function** | MSE: $(y - \hat{y})^2$ | BCE: $-[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ |
| **Error signal** | $-2(y - \hat{y})$ | $(\hat{y} - y)$ |
| **Output range** | $(-\infty, +\infty)$ | $(0, 1)$ — a probability |
| **Learning rate** | Smaller (e.g. `0.0001`) | Can be larger (e.g. `0.01`) |
| **Final decision** | Raw number = prediction | $\hat{y} \geq 0.5$ → class 1, else class 0 |

---

## 6. Key Takeaways

- The **structure of backpropagation is identical** for both problems — you always compute 9 gradients and update 9 parameters. Only the loss function and output activation change.
- Randomly shuffling the dataset each epoch (**stochastic gradient descent**) helps the model generalize and escape local minima.
- Tracking **average loss per epoch** lets you verify the model is actually learning — if loss doesn't decrease, check your learning rate or gradient formulas.
- This linear (no hidden activation) network is a starting point. In Part 3, we add **ReLU or Sigmoid activations on the hidden layer**, which adds one more term to the chain rule for hidden layer gradients.

# Code Example for Regression

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.DataFrame([[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 7]], columns=['cgpa', 'profile_score', 'lpa'])
df

,cgpa,profile_score,lpa
0,8,8,4
1,7,9,5
2,6,10,6
3,5,12,7


In [3]:
def initialize_parameters(layer_dims):
    np.random.seed(42)
    parameters = {}

    L = len(layer_dims)

    for l in range(1, L):
        parameters['W' + str(l)] = np.ones((layer_dims[l-1], layer_dims[l])) * 0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))

    return parameters

In [12]:
initialize_parameters([2, 2, 1])

{'W1': array([[0.1, 0.1],
        [0.1, 0.1]]),
 'b1': array([[0.],
        [0.]]),
 'W2': array([[0.1],
        [0.1]]),
 'b2': array([[0.]])}

In [13]:
def linear_forward(A_prev, W, b):
    return (np.dot(W.T, A_prev) + b)

In [14]:
def forward_propagation(X, parameters):
    A = X
    L = len(parameters) // 2

    for l in range(1, L+1):
        A_prev = A
        Wl = parameters['W' + str(l)]
        bl = parameters['b' + str(l)]
        print("A" + str(l-1) + ": ", A_prev)
        print("W" + str(l-1) + ": ", Wl)
        print("b" + str(l-1) + ": ", bl)
        print("--" * 20)

        A = linear_forward(A_prev, Wl, bl)
        print("A" + str(l) + ": ", A)
        print("**" * 20)

    return A, A_prev

In [15]:
X = df[['cgpa', 'profile_score']].values[0].reshape(2, 1)
y = df[['lpa']].values[0][0]

parameters = initialize_parameters([2, 2, 1])

In [19]:
y_hat, A1 = forward_propagation(X, parameters)

A0:  [[8]
 [8]]
W0:  [[0.1 0.1]
 [0.1 0.1]]
b0:  [[0.]
 [0.]]
----------------------------------------
A1:  [[1.6]
 [1.6]]
****************************************
A1:  [[1.6]
 [1.6]]
W1:  [[0.1]
 [0.1]]
b1:  [[0.]]
----------------------------------------
A2:  [[0.32]]
****************************************


In [17]:
(y - 0.32) ** 2 # loss for first student

np.float64(13.5424)

In [21]:
y_hat = y_hat[0][0]

In [22]:
def update_parameters(parameters, y, y_hat, A1, X):
  parameters['W2'][0][0] = parameters['W2'][0][0] + (0.001 * 2 * (y - y_hat) * A1[0][0])
  parameters['W2'][1][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat) * A1[1][0])
  parameters['b2'][0][0] = parameters['W2'][1][0] + (0.001 * 2 * (y - y_hat))

  parameters['W1'][0][0] = parameters['W1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0] *  X[0][0])
  parameters['W1'][0][1] = parameters['W1'][0][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0] * X[1][0])
  parameters['b1'][0][0] = parameters['b1'][0][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][0][0])

  parameters['W1'][1][0] = parameters['W1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0] * X[0][0])
  parameters['W1'][1][1] = parameters['W1'][1][1] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0] * X[1][0])
  parameters['b1'][1][0] = parameters['b1'][1][0] + (0.001 * 2 * (y - y_hat)*parameters['W2'][1][0])

In [23]:
update_parameters(parameters, y, y_hat, A1, X)

In [24]:
parameters

{'W1': array([[0.10658137, 0.10658137],
        [0.10658137, 0.10658137]]),
 'b1': array([[0.00082267],
        [0.00082267]]),
 'W2': array([[0.111776],
        [0.111776]]),
 'b2': array([[0.119136]])}

In [25]:
X = df[['cgpa', 'profile_score']].values[3].reshape(2,1)
y = df[['lpa']].values[3][0]

In [28]:
y_hat, A1 = forward_propagation(X, parameters)
y_hat = y_hat[0][0]

A0:  [[ 5]
 [12]]
W0:  [[0.10658137 0.10658137]
 [0.10658137 0.10658137]]
b0:  [[0.00082267]
 [0.00082267]]
----------------------------------------
A1:  [[1.81270598]
 [1.81270598]]
****************************************
A1:  [[1.81270598]
 [1.81270598]]
W1:  [[0.111776]
 [0.111776]]
b1:  [[0.119136]]
----------------------------------------
A2:  [[0.52437005]]
****************************************


In [29]:
update_parameters(parameters, y, y_hat, A1, X)

In [30]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 5

for i in range(epochs):

  Loss = []

  for j in range(df.shape[0]):

    X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
    y = df[['lpa']].values[j][0]

    # Parameter initialization
    y_hat, A1 = forward_propagation(X, parameters)
    y_hat = y_hat[0][0]

    update_parameters(parameters, y, y_hat, A1, X)

    Loss.append((y-y_hat)**2)

  print('Epoch - ', i+1, 'Loss - ', np.array(Loss).mean())

parameters

A0:  [[8]
 [8]]
W0:  [[0.1 0.1]
 [0.1 0.1]]
b0:  [[0.]
 [0.]]
----------------------------------------
A1:  [[1.6]
 [1.6]]
****************************************
A1:  [[1.6]
 [1.6]]
W1:  [[0.1]
 [0.1]]
b1:  [[0.]]
----------------------------------------
A2:  [[0.32]]
****************************************
A0:  [[7]
 [9]]
W0:  [[0.10658137 0.10658137]
 [0.10658137 0.10658137]]
b0:  [[0.00082267]
 [0.00082267]]
----------------------------------------
A1:  [[1.70612461]
 [1.70612461]]
****************************************
A1:  [[1.70612461]
 [1.70612461]]
W1:  [[0.111776]
 [0.111776]]
b1:  [[0.119136]]
----------------------------------------
A2:  [[0.50054357]]
****************************************
A0:  [[ 6]
 [10]]
W0:  [[0.11458955 0.1168776 ]
 [0.11458955 0.1168776 ]]
b0:  [[0.0019667]
 [0.0019667]]
----------------------------------------
A1:  [[1.83539945]
 [1.87200826]]
****************************************
A1:  [[1.83539945]
 [1.87200826]]
W1:  [[0.12712927]
 [0.127

{'W1': array([[0.26507636, 0.38558861],
        [0.27800387, 0.40980287]]),
 'b1': array([[0.02749056],
        [0.02974394]]),
 'W2': array([[0.41165744],
        [0.48302736]]),
 'b2': array([[0.48646246]])}

# Code Example for Classification

In [31]:
df = pd.DataFrame([[8,8,1], [7,9,1], [6,10,0], [5,5,0]], columns=['cgpa', 'profile_score', 'placed'])

In [32]:
def sigmoid(Z):
    return (1 / (1 + np.exp(-Z)))

In [33]:
def linear_forward(A_prev, W, b):
    Z = np.dot(W.T, A_prev) + b
    return sigmoid(Z)

In [34]:
# L-layer feed forward
def L_layer_forward(X, parameters):
    A = X
    L = len(parameters) // 2                  # number of layers in the neural network

    for l in range(1, L+1):
        A_prev = A 
        Wl = parameters['W' + str(l)]
        bl = parameters['b' + str(l)]
        #print("A"+str(l-1)+": ", A_prev)
        #print("W"+str(l)+": ", Wl)
        #print("b"+str(l)+": ", bl)
        #print("--"*20)

        A = linear_forward(A_prev, Wl, bl)
        #print("A"+str(l)+": ", A)
        #print("**"*20)
            
    return A,A_prev


In [35]:
def update_parameters(parameters,y,y_hat,A1,X):
  parameters['W2'][0][0] = parameters['W2'][0][0] + (0.0001 * (y - y_hat) * A1[0][0])
  parameters['W2'][1][0] = parameters['W2'][1][0] + (0.0001 * (y - y_hat) * A1[1][0])
  parameters['b2'][0][0] = parameters['W2'][1][0] + (0.0001 * (y - y_hat))

  parameters['W1'][0][0] = parameters['W1'][0][0] + (0.0001 * (y - y_hat) * parameters['W2'][0][0] * A1[0][0] * (1 - A1[0][0]) * X[0][0])
  parameters['W1'][0][1] = parameters['W1'][0][1] + (0.0001 * (y - y_hat) * parameters['W2'][0][0] * A1[0][0] * (1 - A1[0][0]) * X[1][0])
  parameters['b1'][0][0] = parameters['b1'][0][0] + (0.0001 * (y - y_hat) * parameters['W2'][0][0] * A1[0][0] * (1 - A1[0][0]))

  parameters['W1'][1][0] = parameters['W1'][1][0] + (0.0001 * (y - y_hat) * parameters['W2'][1][0] * A1[1][0] * (1 - A1[1][0]) * X[0][0])
  parameters['W1'][1][1] = parameters['W1'][1][1] + (0.0001 * (y - y_hat) * parameters['W2'][1][0] * A1[1][0] * (1 - A1[1][0]) * X[1][0])
  parameters['b1'][1][0] = parameters['b1'][1][0] + (0.0001 * (y - y_hat) * parameters['W2'][1][0] * A1[1][0] * (1 - A1[1][0]))

In [36]:
X = df[['cgpa', 'profile_score']].values[0].reshape(2,1) # Shape(no of features, no. of training example)
y = df[['placed']].values[0][0]

# Parameter initialization
parameters = initialize_parameters([2,2,1])

y_hat,A1 = L_layer_forward(X, parameters)
y_hat = y_hat[0][0]

update_parameters(parameters, y, y_hat, A1, X)

print('Loss for this student - ', -y * np.log(y_hat) - (1 - y) * np.log(1 - y_hat))

parameters

Loss for this student -  0.613402628898913


{'W1': array([[0.10000513, 0.10000513],
        [0.10000513, 0.10000513]]),
 'b1': array([[6.41054186e-07],
        [6.41054186e-07]]),
 'W2': array([[0.10003815],
        [0.10003815]]),
 'b2': array([[0.100084]])}

In [37]:
# epochs implementation

parameters = initialize_parameters([2,2,1])
epochs = 50

for i in range(epochs):

  Loss = []

  for j in range(df.shape[0]):

    X = df[['cgpa', 'profile_score']].values[j].reshape(2,1) # Shape(no of features, no. of training example)
    y = df[['placed']].values[j][0]

    # Parameter initialization


    y_hat, A1 = L_layer_forward(X, parameters)
    y_hat = y_hat[0][0]

    update_parameters(parameters, y, y_hat, A1, X)

    Loss.append(-y * np.log(y_hat) - (1 - y) * np.log(1 - y_hat))

  print('Epoch - ', i+1, 'Loss - ', np.array(Loss).mean())

parameters

Epoch -  1 Loss -  0.7103199085929446
Epoch -  2 Loss -  0.6991702892802629
Epoch -  3 Loss -  0.6991679314811485
Epoch -  4 Loss -  0.6991655746710999
Epoch -  5 Loss -  0.6991632188496667
Epoch -  6 Loss -  0.699160864016399
Epoch -  7 Loss -  0.6991585101708473
Epoch -  8 Loss -  0.6991561573125619
Epoch -  9 Loss -  0.6991538054410936
Epoch -  10 Loss -  0.6991514545559935
Epoch -  11 Loss -  0.6991491046568126
Epoch -  12 Loss -  0.6991467557431024
Epoch -  13 Loss -  0.6991444078144144
Epoch -  14 Loss -  0.6991420608703007
Epoch -  15 Loss -  0.6991397149103132
Epoch -  16 Loss -  0.6991373699340042
Epoch -  17 Loss -  0.6991350259409265
Epoch -  18 Loss -  0.6991326829306324
Epoch -  19 Loss -  0.6991303409026751
Epoch -  20 Loss -  0.699127999856608
Epoch -  21 Loss -  0.6991256597919842
Epoch -  22 Loss -  0.6991233207083575
Epoch -  23 Loss -  0.6991209826052818
Epoch -  24 Loss -  0.699118645482311
Epoch -  25 Loss -  0.6991163093389996
Epoch -  26 Loss -  0.699113974174902

{'W1': array([[0.09994267, 0.09984548],
        [0.09994272, 0.09984548]]),
 'b1': array([[-3.38405750e-05],
        [-3.38419977e-05]]),
 'W2': array([[0.09920806],
        [0.09920816]]),
 'b2': array([[0.09915209]])}